# Feature Extraction — YAMNet embeddings (D1 & D2)

Notebook ini menjalankan **YAMNet beku** sebagai feature extractor: tiap file audio
diringkas jadi satu vektor **1024-d**. Vektor ini di-**cache** ke `ml/cache/`, lalu
notebook training tinggal memuatnya — YAMNet tidak perlu dijalankan ulang tiap eksperimen.

## Kenapa begini

- YAMNet sudah belajar representasi suara dari jutaan klip AudioSet. Kita **tidak melatih
  ulang** bobotnya (butuh GPU + data besar); cukup pakai embedding-nya sebagai ringkasan.
- Di atas embedding, notebook berikutnya melatih classifier kecil (ringan, cukup CPU).
- Satu notebook untuk **kedua dataset** — memuat YAMNet mahal, jadi dilakukan sekali.

## Detail teknis penting

| Hal | Nilai | Catatan |
|---|---|---|
| Sample rate | **16000 Hz** | YAMNet mewajibkan 16k — beda dari 22050 di preprocessing |
| Durasi | 3.0 s → 48000 sampel | pad/crop supaya seragam |
| Output YAMNet | ~6 frame × 1024-d | di-**mean-pool** jadi 1 vektor 1024-d/file |
| Keras | **legacy (Keras 2)** | `TF_USE_LEGACY_KERAS=1` wajib sebelum `import tensorflow` |

> Manifest & split bersifat representation-agnostic, jadi resample ke 16k di sini tidak
> bertentangan dengan preprocessing — cuma representasi berbeda dari sumber yang sama.

In [1]:
# --- WAJIB: set sebelum tensorflow di-import (lihat CLAUDE.md) ---
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")     # kurangi spam log TF

import time
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------------ config
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ML_DIR = ROOT / "ml"
CACHE_DIR = ML_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

YAMNET_SR = 16000                                      # YAMNet mewajibkan 16 kHz
DURATION = 3.0
N_SAMPLES = int(YAMNET_SR * DURATION)                  # 48000

DATASETS = {
    "d1": ML_DIR / "manifest_d1.csv",
    "d2": ML_DIR / "manifest_d2.csv",
}

print(f"tensorflow {tf.__version__} · keras legacy = {os.environ['TF_USE_LEGACY_KERAS']}")
print(f"cache -> {CACHE_DIR}")
for name, m in DATASETS.items():
    assert m.exists(), f"manifest {name} belum ada — jalankan notebook preprocessing dulu: {m}"
print("manifest D1 & D2 ditemukan.")

tensorflow 2.16.2 · keras legacy = 1
cache -> D:\Coding Vscode\Siren Classification\ml\cache
manifest D1 & D2 ditemukan.


---
## 1 · Muat YAMNet

Diunduh sekali dari TF-Hub lalu di-cache oleh TF-Hub sendiri (unduhan berikutnya instan).

In [2]:
t0 = time.time()
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
print(f"YAMNet dimuat ({time.time() - t0:.1f}s)")

# sanity check bentuk output pada 3 detik senyap
_scores, _emb, _spec = yamnet(np.zeros(N_SAMPLES, dtype=np.float32))
print(f"contoh output: scores {_scores.shape} · embeddings {_emb.shape}")
EMB_DIM = int(_emb.shape[-1])
print(f"dimensi embedding = {EMB_DIM}")

YAMNet dimuat (2.2s)
contoh output: scores (6, 521) · embeddings (6, 1024)
dimensi embedding = 1024


---
## 2 · Fungsi ekstraksi

`load_16k` menyiapkan audio (16k, mono, peak-normalize, panjang tetap). `embed`
menjalankan YAMNet lalu **mean-pool** embedding antar-frame jadi satu vektor per file.

> Mean-pool dipilih karena sederhana dan efektif untuk klip pendek homogen seperti ini.
> Alternatif (mis. gabung mean+std, atau simpan per-frame) bisa dicoba nanti kalau perlu.

In [3]:
def load_16k(path: str) -> np.ndarray:
    """Audio siap-YAMNet: 16 kHz, mono, peak-normalized, panjang tetap N_SAMPLES."""
    y, _ = librosa.load(path, sr=YAMNET_SR, mono=True)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    y = y[:N_SAMPLES].astype(np.float32)
    peak = np.abs(y).max()
    return y / peak if peak > 0 else y


def embed(path: str) -> np.ndarray:
    """Satu vektor 1024-d per file (mean-pool embedding antar-frame YAMNet)."""
    _scores, emb, _spec = yamnet(load_16k(path))
    return emb.numpy().mean(axis=0)


# validasi pada 1 file
sample_path = ROOT / pd.read_csv(DATASETS["d1"]).iloc[0].path
v = embed(str(sample_path))
assert v.shape == (EMB_DIM,), f"bentuk embedding salah: {v.shape}"
print(f"validasi ok — embedding {v.shape} dari {sample_path.name}")

validasi ok — embedding (1024,) dari sound_1.wav


---
## 3 · Ekstrak & cache kedua dataset

Untuk tiap dataset: baca manifest, ekstrak embedding semua file (urutan manifest
dipertahankan), simpan ke `ml/cache/yamnet_<name>.npz` berisi `emb, filename, label,
source_id`. Urutan yang dipertahankan inilah yang nanti dipakai notebook training untuk
mencocokkan embedding dengan baris split lewat `filename`.

In [4]:
def extract_dataset(name: str, manifest_path: Path) -> dict:
    df = pd.read_csv(manifest_path)
    n = len(df)
    embs = np.zeros((n, EMB_DIM), dtype=np.float32)

    print(f"[{name}] {n} file ...")
    t0 = time.time()
    for i, row in enumerate(df.itertuples(index=False)):
        embs[i] = embed(str(ROOT / row.path))
        if (i + 1) % 250 == 0 or i + 1 == n:
            rate = (i + 1) / (time.time() - t0)
            print(f"  {i + 1:>4}/{n}  ({rate:.0f} file/s)")

    out = CACHE_DIR / f"yamnet_{name}.npz"
    np.savez(
        out,
        emb=embs,
        filename=df.filename.to_numpy(),
        label=df.label.to_numpy(),
        source_id=df.source_id.to_numpy(),
    )
    print(f"[{name}] tersimpan -> {out}  ({embs.shape})  {time.time() - t0:.0f}s\n")
    return {"name": name, "shape": embs.shape, "path": out}


results = [extract_dataset(name, path) for name, path in DATASETS.items()]

[d1] 596 file ...


   250/596  (69 file/s)


   500/596  (69 file/s)


   596/596  (68 file/s)
[d1] tersimpan -> D:\Coding Vscode\Siren Classification\ml\cache\yamnet_d1.npz  ((596, 1024))  9s

[d2] 1675 file ...


   250/1675  (66 file/s)


...

   750/1675  (66 file/s)


  1000/1675  (66 file/s)


  1250/1675  (66 file/s)


  1500/1675  (66 file/s)


  1675/1675  (65 file/s)
[d2] tersimpan -> D:\Coding Vscode\Siren Classification\ml\cache\yamnet_d2.npz  ((1675, 1024))  26s



---
## 4 · Verifikasi cache

Muat kembali file cache, pastikan bentuknya benar dan `filename`-nya cocok dengan
split CSV (jumlah baris per split saat di-join harus utuh — tidak ada file yang hilang).

In [5]:
for name in DATASETS:
    data = np.load(CACHE_DIR / f"yamnet_{name}.npz", allow_pickle=True)
    emb, fnames = data["emb"], data["filename"]
    print(f"[{name}] emb {emb.shape} · {len(fnames)} filename · "
          f"NaN={np.isnan(emb).any()}")

    # cek tiap split ketemu semua embedding-nya
    cached = set(fnames)
    for split in ("train", "val", "test"):
        sp = pd.read_csv(ML_DIR / f"split_{name}_{split}.csv")
        missing = set(sp.filename) - cached
        assert not missing, f"{name}/{split}: {len(missing)} file tak ada di cache!"
        print(f"    {split:5s} {len(sp):>4} file  -> semua ada di cache")
print("\nOK — cache lengkap & konsisten dengan split.")

[d1] emb (596, 1024) · 596 filename · NaN=False
    train  426 file  -> semua ada di cache
    val     85 file  -> semua ada di cache
    test    85 file  -> semua ada di cache
[d2] emb (1675, 1024) · 1675 filename · NaN=False
    train 1195 file  -> semua ada di cache
    val    240 file  -> semua ada di cache
    test   240 file  -> semua ada di cache

OK — cache lengkap & konsisten dengan split.


---

**Selanjutnya:** `05_train_dataset1.ipynb` & `05_train_dataset2.ipynb` memuat embedding
dari cache, mencocokkannya dengan split, lalu melatih classifier kecil dan mengukur
macro-F1. YAMNet tidak disentuh lagi.